In [1]:
from typing import List

In [2]:
class Solution:

    def __init__(self):
        

        self.board = None
        self.rows = [set() for _ in range(9)]
        self.cols = [set() for _ in range(9)]
        self.mids = [set() for _ in range(9)]
        self.num = 0
    
        self.full = set([str(i + 1) for i in range(9)])
        self.stack = list() # (i, j, num) triplets

    def is_full(self):
        return self.num == 81

    def mid_index(self, i, j):
        return (i // 3) * 3 + (j // 3)

    def print_me(self):
        for i in range(9):
            for j in range(9):
                print(self.board[i][j], end=" ")
            print()

    def add(self, i, j, value):
        # print(f"adding {value} to {i},{j} cell...")

        mid_i = self.mid_index(i, j)

        if self.board[i][j] != "." or value in self.rows[i] or value in self.cols[j] or value in self.mids[mid_i]:
            return False
        
        if value not in self.rows[i]:
            self.rows[i].add(value)
            self.stack.append((i, value, "rows"))
        if value not in self.cols[j]:
            self.cols[j].add(value)
            self.stack.append((j, value, "cols"))
        
        if value not in self.mids[mid_i]:
            self.mids[mid_i].add(value)
            self.stack.append((mid_i, value, "mids"))

        self.board[i][j] = value
        self.stack.append(((i, j), value, "board"))

        self.num += 1

        return True

    def remove(self, i, j, value):
        # print(f"adding {value} to {i},{j} cell...")
        self.board[i][j] = "."

        
        mid_i = self.mid_index(i, j)
        self.rows[i].remove(value)
        self.cols[j].remove(value)
        self.mids[mid_i].remove(value)

        self.num -= 1

    def remove_instruction(self):
        index, value, op = self.stack.pop()
        if op == "board":
            i, j = index
            self.board[i][j] = "."
            self.num -= 1
        else:
            target_set = self.mids[index]
            if op == "rows":
                target_set = self.rows[index]
            elif op == "cols":
                target_set = self.cols[index]
            target_set.remove(value)

    def base_solve(self):
        while True:
            found = False
            for i in range(9):
                for j in range(9):
                    if self.board[i][j] == ".":
                        mid_i = self.mid_index(i, j)
                        candidates = self.full - (self.rows[i] | self.cols[j] | self.mids[mid_i])
                        # print(f"candidates for {i},{j}: {candidates}, excluded_set={excluded_set}")
                        if len(candidates) == 1:
                            elem = candidates.pop()
                            # print(f". {elem} qualified for {i},{j} because no other possibilities exist")
                            if not self.add(i, j, elem):
                                return
                            found = True
            
            if self.is_full():
                return
            
            for i in range(9):
                for j in range(9):
                    mid_i = self.mid_index(i, j)
                    if self.board[i][j] == ".":
                        candidates = self.full - (self.rows[i] | self.cols[j] | self.mids[mid_i])
                        if len(candidates) > 0:
                            for elem in candidates:
                                # All other row elems
                                all_others_excluded = True
                                for k in range(9):
                                    if k != j and self.board[i][k] == ".":
                                        if elem not in self.cols[k] and elem not in self.mids[self.mid_index(i, k)]:
                                            all_others_excluded = False
                                            break
                                if all_others_excluded:
                                    # print(f". {elem} qualified for {i},{j} because all other row elems are excluded")
                                    if not self.add(i, j, elem):
                                        return
                                    found = True
                                    break
                                
                                # All other col elems
                                all_others_excluded = True
                                for k in range(9):
                                    if k != i and self.board[k][j] == ".":
                                        if elem not in self.rows[k] and elem not in self.mids[self.mid_index(k, j)]:
                                            all_others_excluded = False
                                            break
                                
                                if all_others_excluded:
                                    # print(f". {elem} qualified for {i},{j} because all other col elems are excluded")
                                    if not self.add(i, j, elem):
                                        return
                                    found = True
                                    break
                                
                                # All other mid elements
                                all_others_excluded = True
                                cell_row_start = (i // 3) * 3
                                cell_col_start = (j // 3) * 3
                                for k in range(cell_row_start, cell_row_start + 3):
                                    for s in range(cell_col_start, cell_col_start + 3):
                                        if k == i and s == j:
                                            continue
                                        if self.board[k][s] == ".":
                                            if elem not in self.rows[k] and elem not in self.cols[s]:
                                                all_others_excluded = False
                                                break
                                    if not all_others_excluded:
                                        break
                                
                                if all_others_excluded:
                                    # print(f". {elem} qualified for {i},{j} because all other cell elems are excluded")
                                    if not self.add(i, j, elem):
                                        return
                                    found = True
            
            if not found:
                break
        
        self.print_me()


    def rec_solve(self):
        if self.is_full():
            return True
        
        min_num = 9
        min_candidates = set()
        min_i, min_j = -1, -1
        for i in range(9):
            for j in range(9):
                if self.board[i][j] == ".":
                    candidates = self.full - (self.rows[i] | self.cols[j] | self.mids[self.mid_index(i, j)])
                    if len(candidates) < min_num:
                        min_num = len(candidates)
                        min_candidates = candidates
                        min_i, min_j = i, j
                        if len(min_candidates) == 2:
                            break
            if len(min_candidates) == 2:
                break
        
        for elem in min_candidates:
            # stack_len_before = len(self.stack)
            print(f"Assuming {min_i},{min_j} is {elem} from its set of {min_candidates}")
            instructions_before = len(self.stack)
            self.add(min_i, min_j, elem)
            self.base_solve()
            solved = self.rec_solve()
            if solved:
                return True
                
            while len(self.stack) > instructions_before:
                self.remove_instruction()
        
        return False

    def solveSudoku(self, board: List[List[str]]) -> None:
        """
        Do not return anything, modify board in-place instead.
        """
        for i in range(9):
            for j in range(9):
                elem = board[i][j]
                if elem != ".":
                    mid_i = self.mid_index(i, j)
                    self.rows[i].add(elem)
                    self.cols[j].add(elem)
                    self.mids[mid_i].add(elem)
                    self.num += 1

        self.board = board

        self.base_solve() # solve as much as you can without backtracking
        self.rec_solve()

In [3]:
soln = Solution()

In [4]:
board = [
    [".", ".", "9", "7", "4", "8", ".", ".", "."],
    ["7", ".", ".", ".", ".", ".", ".", ".", "."],
    [".", "2", ".", "1", ".", "9", ".", ".", "."],
    [".", ".", "7", ".", ".", ".", "2", "4", "."],
    [".", "6", "4", ".", "1", ".", "5", "9", "."],
    [".", "9", "8", ".", ".", ".", "3", ".", "."],
    [".", ".", ".", "8", ".", "3", ".", "2", "."],
    [".", ".", ".", ".", ".", ".", ".", ".", "6"],
    [".", ".", ".", "2", "7", "5", "9", ".", "."],
]

In [5]:
soln.num

0

In [6]:
soln.solveSudoku(board)

. . 9 7 4 8 . . 2 
7 . . 6 . 2 . . 9 
. 2 . 1 . 9 . . . 
. . 7 9 8 6 2 4 1 
2 6 4 3 1 7 5 9 8 
1 9 8 5 2 4 3 6 7 
9 . . 8 6 3 . 2 . 
. . 2 4 9 1 . . 6 
. . . 2 7 5 9 . . 
Assuming 0,6 is 6 from its set of {'6', '1'}
. . 9 7 4 8 6 . 2 
7 . . 6 . 2 . . 9 
. 2 . 1 . 9 . . . 
. . 7 9 8 6 2 4 1 
2 6 4 3 1 7 5 9 8 
1 9 8 5 2 4 3 6 7 
9 . . 8 6 3 . 2 . 
. . 2 4 9 1 . . 6 
. . . 2 7 5 9 . . 
Assuming 0,0 is 3 from its set of {'3', '5'}
3 1 9 7 4 8 6 5 2 
7 8 . 6 3 2 4 1 9 
4 2 6 1 5 9 8 7 3 
5 3 7 9 8 6 2 4 1 
2 6 4 3 1 7 5 9 8 
1 9 8 5 2 4 3 6 7 
9 7 5 8 6 3 1 2 4 
8 . 2 4 9 1 7 3 6 
6 4 3 2 7 5 9 8 . 
Assuming 0,0 is 5 from its set of {'3', '5'}


In [7]:
soln.print_me()

5 1 9 7 4 8 6 3 2 
7 8 3 6 5 2 4 1 9 
4 2 6 1 3 9 8 7 5 
3 5 7 9 8 6 2 4 1 
2 6 4 3 1 7 5 9 8 
1 9 8 5 2 4 3 6 7 
9 7 5 8 6 3 1 2 4 
8 3 2 4 9 1 7 5 6 
6 4 1 2 7 5 9 8 3 


In [8]:
soln.board

[['5', '1', '9', '7', '4', '8', '6', '3', '2'],
 ['7', '8', '3', '6', '5', '2', '4', '1', '9'],
 ['4', '2', '6', '1', '3', '9', '8', '7', '5'],
 ['3', '5', '7', '9', '8', '6', '2', '4', '1'],
 ['2', '6', '4', '3', '1', '7', '5', '9', '8'],
 ['1', '9', '8', '5', '2', '4', '3', '6', '7'],
 ['9', '7', '5', '8', '6', '3', '1', '2', '4'],
 ['8', '3', '2', '4', '9', '1', '7', '5', '6'],
 ['6', '4', '1', '2', '7', '5', '9', '8', '3']]

In [ ]:
soln.stack